In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import time
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [21]:
column_names = [
    "mpg", "cylinders", "displacement", "horsepower",
    "weight", "acceleration", "model_year", "origin", "name"
]

df = pd.read_csv(
    "./auto+mpg/auto-mpg.data",
    names=column_names,
    na_values="?",
    sep="\s+",
)

df.head()

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
C:\Users\gerar\AppData\Local\Temp\ipykernel_2184\2718336478.py:10: SyntaxWarning: invalid escape sequence '\s'
  sep="\s+",


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino


In [22]:
df = df.dropna()

# eliminar columna no útil
df = df.drop(columns=["name"])

df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1


In [23]:
X = df.drop("mpg", axis=1).values
y = df["mpg"].values.reshape(-1, 1)

In [24]:
print(df.shape)
df.head()

(392, 8)


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1


In [25]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

In [26]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)

X_val = torch.FloatTensor(X_val)
y_val = torch.FloatTensor(y_val)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

In [27]:
input_size = X.shape[1]
learning_rate = 0.01
num_epochs = 20

In [28]:
class Net1(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 1)
        )

    def forward(self, x):
        return self.model(x)


class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 10),
            nn.ReLU(),
            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.model(x)


class Net3(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 10),
            nn.ReLU(),
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 30),
            nn.ReLU(),
            nn.Linear(30, 20),
            nn.ReLU(),
            nn.Linear(20, 1)
        )

    def forward(self, x):
        return self.model(x)

In [29]:
def train_model(model, train_loader, val_loader, num_epochs, learning_rate):
    model = model
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)

    val_losses = []
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()

        for X_batch, y_batch in train_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] - Val Loss: {val_loss:.4f}")

    training_time = time.time() - start_time
    return model, val_losses, training_time

In [30]:
results = {}

model1 = Net1()
model1, val1, time1 = train_model(model1, train_loader, val_loader, num_epochs, learning_rate)
results["Network 1"] = {"val_loss": val1[-1], "time": time1}

model2 = Net2()
model2, val2, time2 = train_model(model2, train_loader, val_loader, num_epochs, learning_rate)
results["Network 2"] = {"val_loss": val2[-1], "time": time2}

model3 = Net3()
model3, val3, time3 = train_model(model3, train_loader, val_loader, num_epochs, learning_rate)
results["Network 3"] = {"val_loss": val3[-1], "time": time3}

Epoch [1/20] - Val Loss: 0.5389


Epoch [2/20] - Val Loss: 0.2994
Epoch [3/20] - Val Loss: 0.2698
Epoch [4/20] - Val Loss: 0.2589
Epoch [5/20] - Val Loss: 0.2489
Epoch [6/20] - Val Loss: 0.2383
Epoch [7/20] - Val Loss: 0.2302
Epoch [8/20] - Val Loss: 0.2222
Epoch [9/20] - Val Loss: 0.2168
Epoch [10/20] - Val Loss: 0.2156
Epoch [11/20] - Val Loss: 0.2143
Epoch [12/20] - Val Loss: 0.2165
Epoch [13/20] - Val Loss: 0.2162
Epoch [14/20] - Val Loss: 0.2165
Epoch [15/20] - Val Loss: 0.2141
Epoch [16/20] - Val Loss: 0.2166
Epoch [17/20] - Val Loss: 0.2190
Epoch [18/20] - Val Loss: 0.2166
Epoch [19/20] - Val Loss: 0.2171
Epoch [20/20] - Val Loss: 0.2140
Epoch [1/20] - Val Loss: 0.7290
Epoch [2/20] - Val Loss: 0.7068
Epoch [3/20] - Val Loss: 0.6821
Epoch [4/20] - Val Loss: 0.6530
Epoch [5/20] - Val Loss: 0.6159
Epoch [6/20] - Val Loss: 0.5681
Epoch [7/20] - Val Loss: 0.5073
Epoch [8/20] - Val Loss: 0.4408
Epoch [9/20] - Val Loss: 0.3768
Epoch [10/20] - Val Loss: 0.3236
Epoch [11/20] - Val Loss: 0.2853
Epoch [12/20] - Val Loss: 0

In [31]:
for name, info in results.items():
    print(name)
    print(f"  Validation Loss: {info['val_loss']:.4f}")
    print(f"  Training Time: {info['time']:.2f} seconds")
    print()

Network 1
  Validation Loss: 0.2140
  Training Time: 0.30 seconds

Network 2
  Validation Loss: 0.2357
  Training Time: 0.42 seconds

Network 3
  Validation Loss: 0.6765
  Training Time: 0.37 seconds



In [32]:
best_model = min(results, key=lambda x: results[x]["val_loss"])

print("Best model:")
print(best_model)
print(f"Validation Loss: {results[best_model]['val_loss']:.4f}")

Best model:
Network 1
Validation Loss: 0.2140


## Answer

The model with the least validation error was **Network 1**, with a final validation loss of **0.2140**.

Training times:
- Network 1: **0.30 seconds**
- Network 2: **0.42 seconds**
- Network 3: **0.37 seconds**